# 01 - Sign and query

Sign a claim and a support, then query the discourse edge across the two signed
files. Local only, nothing is published. Profile setup is in the README.

In [1]:
# Setup. Local only
import sys, os, subprocess, glob
try:
    import nanopub, rdflib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nanopub", "rdflib"])
sys.path.insert(0, os.getcwd())
try:
    import na_nanopub as na
except ModuleNotFoundError:
    hits = glob.glob(os.path.join(os.getcwd(), "**", "na_nanopub.py"), recursive=True)
    if hits:
        sys.path.insert(0, os.path.dirname(hits[0]))
    import na_nanopub as na
import nanopub as _np
print("nanopub", _np.__version__, "-- ready, signing locally (no network)")

nanopub 2.1.0 -- ready, signing locally (no network)


## Root claim

`sub:claim` becomes `<trusty>/claim` at sign time.

In [2]:
root = na.make(
    '''  sub:claim a schema:Statement ;
    rdf:value             "Certain gut microbiota profiles have stronger mRNA vaccine responses." ;
    cito:citesAsEvidence  <https://doi.org/10.1234/kim-2025-microbiota-vaccine> .''',
    attributed_to="orcid:0000-0001-alvarez-b",
    created="2026-05-24T08:00:00Z",
    nanopub_type="schema:Statement",
    introduces="sub:claim", name="01-root",
)
na.show(root, "root")
claim = root.source_uri + "/claim"
print("content node:", claim)

ERROR! Session/line number was not unique in database. History logging moved to new session 73
root           https://w3id.org/np/RAIxUlZOlAthR-D__aPCP9ELfpK0M1zfFpi6zbucr295Q   [Statement]
content node: https://w3id.org/np/RAIxUlZOlAthR-D__aPCP9ELfpK0M1zfFpi6zbucr295Q/claim


## Support

In [3]:
support = na.make(
    f'''  sub:reason a schema:Statement ;
    rdf:value      "Their SCFA data is compelling." ;
    cito:supports  <{claim}> .''',
    attributed_to="orcid:0000-0002-wang-p",
    created="2026-05-24T09:00:00Z",
    nanopub_type="cito:supports",
    introduces="sub:reason", name="01-support",
)
na.show(support, "support")

support        https://w3id.org/np/RAMHN_J65Na8c-2VDptNnH_gxefvf6W9hWZMESYYN0-04   [supports]


## Query: what targets the claim?

Runs over the two signed files merged in memory.

In [4]:
ds = na.load(root, support)
rows = na.targets(ds, claim)
print(f"{len(rows)} contribution(s) target {claim}:\n")
for r in rows:
    print(f"  {na.localname(r['relation']):12s} <- {na.localname(r['contribution'])}")
    print(f"      \"{r['value']}\"")

1 contribution(s) target https://w3id.org/np/RAIxUlZOlAthR-D__aPCP9ELfpK0M1zfFpi6zbucr295Q/claim:

  supports     <- reason
      "Their SCFA data is compelling."
